In [ ]:
import panel as pn
import pandas as pd
from config.database import SessionLocal
from models import Cuidador

pn.extension('tabulator')

status_alert = pn.pane.Alert("", alert_type="light", visible=False)

txt_cpf = pn.widgets.TextInput(name="CPF Cuidador")
txt_nome = pn.widgets.TextInput(name="Nome Completo")
txt_dias = pn.widgets.TextInput(name="Dias Disponíveis (Ex: Seg a Sex)")
txt_horas = pn.widgets.TextInput(name="Horários (Ex: 08h-18h)")

btn_salvar = pn.widgets.Button(name="Salvar/Atualizar", button_type="success")
btn_excluir = pn.widgets.Button(name="Excluir Selecionado", button_type="danger")

tabela_dados = pn.widgets.Tabulator(pd.DataFrame(), height=300, layout='fit_columns', selectable=True)

def carregar_dados():
    db = SessionLocal()
    try:
        query = db.query(Cuidador).all()
        dados = [{"CPF": c.cpf_cuidador, "Nome": c.nome_completo, "Dias": c.dias_disponiveis, "Horas": c.horario_disponivel} for c in query]
        tabela_dados.value = pd.DataFrame(dados)
    finally:
        db.close()

def salvar_registro(event):
    if not txt_cpf.value or not txt_nome.value:
        status_alert.object = "CPF e Nome são obrigatórios."
        status_alert.alert_type = "danger"
        status_alert.visible = True
        return

    db = SessionLocal()
    try:
        cuidador = db.query(Cuidador).filter_by(cpf_cuidador=txt_cpf.value).first()
        if not cuidador:
            cuidador = Cuidador(cpf_cuidador=txt_cpf.value)
            db.add(cuidador)
            msg = "Cuidador cadastrado!"
        else:
            msg = "Cuidador atualizado!"
            
        cuidador.nome_completo = txt_nome.value
        cuidador.dias_disponiveis = txt_dias.value
        cuidador.horario_disponivel = txt_horas.value
        
        db.commit()
        status_alert.object = msg
        status_alert.alert_type = "success"
        status_alert.visible = True
        carregar_dados()
    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro: {str(e)}"
        status_alert.alert_type = "danger"
        status_alert.visible = True
    finally:
        db.close()

def excluir_registro(event):
    selecao = tabela_dados.selection
    if not selecao:
        status_alert.object = "Selecione uma linha para excluir."
        status_alert.alert_type = "warning"
        status_alert.visible = True
        return
    
    cpf_selecionado = tabela_dados.value.iloc[selecao[0]]['CPF']
    db = SessionLocal()
    try:
        cuidador = db.query(Cuidador).filter_by(cpf_cuidador=cpf_selecionado).first()
        if cuidador:
            db.delete(cuidador)
            db.commit()
            status_alert.object = "Excluído com sucesso!"
            status_alert.alert_type = "success"
            status_alert.visible = True
            carregar_dados()
    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro: {str(e)}"
        status_alert.alert_type = "danger"
        status_alert.visible = True
    finally:
        db.close()

btn_salvar.on_click(salvar_registro)
btn_excluir.on_click(excluir_registro)

dashboard = pn.Column(
    "## Gestão de Cuidadores", status_alert,
    pn.Row(txt_cpf, txt_nome), pn.Row(txt_dias, txt_horas),
    pn.Row(btn_salvar, btn_excluir),
    "### Cuidadores Cadastrados", tabela_dados
)
carregar_dados()
dashboard.servable()